loding the data from previous notebook of the baslinfeature

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed/datasets")

X_train = pd.read_parquet(DATA_DIR / "X_train_baseline.parquet")
y_train = pd.read_parquet(DATA_DIR / "y_train.parquet")["lithology"]

X_public = pd.read_parquet(DATA_DIR / "X_public_baseline.parquet")
y_public = pd.read_parquet(DATA_DIR / "y_public.parquet")["lithology"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_public:", X_public.shape)
print("y_public:", y_public.shape)

print("\nClasses:", sorted(y_train.unique()))
print("Number of classes:", y_train.nunique())

X_train: (1171557, 5)
y_train: (1171557,)
X_public: (136841, 5)
y_public: (136841,)

Classes: [np.float64(30000.0), np.float64(65000.0), np.float64(65030.0), np.float64(70000.0), np.float64(70032.0), np.float64(74000.0), np.float64(80000.0), np.float64(86000.0), np.float64(88000.0), np.float64(90000.0), np.float64(93000.0), np.float64(99000.0)]
Number of classes: 12


**our first model is random forest**

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)
import time

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

print("Training Random Forest...")

start = time.perf_counter()

rf_baseline.fit(X_train, y_train)

train_time = time.perf_counter() - start

print(f"Training time: {train_time:.2f} seconds")

Training Random Forest...
Training time: 75.81 seconds


**now evaluating this model on the 10 public well**

In [5]:
print("Predicting PUBLIC set...")

start = time.perf_counter()

y_public_pred = rf_baseline.predict(X_public)

prediction_time = time.perf_counter() - start

accuracy = accuracy_score(y_public, y_public_pred)
balanced_acc = balanced_accuracy_score(y_public, y_public_pred)
macro_f1 = f1_score(y_public, y_public_pred, average="macro")
weighted_f1 = f1_score(y_public, y_public_pred, average="weighted")

print(f"Prediction time: {prediction_time:.2f} seconds")
print(f"Accuracy:          {accuracy:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"Macro F1:          {macro_f1:.4f}")
print(f"Weighted F1:       {weighted_f1:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_public,
        y_public_pred,
        zero_division=0
    )
)

Predicting PUBLIC set...
Prediction time: 0.83 seconds
Accuracy:          0.5289
Balanced Accuracy: 0.4237
Macro F1:          0.2718
Weighted F1:       0.5922

Classification report:
              precision    recall  f1-score   support

     30000.0       0.74      0.71      0.72     24048
     65000.0       0.88      0.52      0.65     84030
     65030.0       0.29      0.38      0.33     17558
     70000.0       0.35      0.49      0.41      4798
     70032.0       0.35      0.18      0.24       625
     74000.0       0.00      0.01      0.00       416
     80000.0       0.08      0.36      0.13      3306
     86000.0       0.49      0.22      0.30       125
     88000.0       0.00      0.00      0.00         0
     90000.0       0.25      0.71      0.37       690
     93000.0       0.00      0.00      0.00         0
     99000.0       0.06      0.66      0.10      1245

    accuracy                           0.53    136841
   macro avg       0.29      0.35      0.27    136841
weigh

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Metric | Public score |
|---|---:|
| Accuracy | **0.5289** |
| Balanced Accuracy | **0.4237** |
| Macro F1 | **0.2718** |
| Weighted F1 | **0.5922** |

**model is doing reasonably well on common lithologies:**
- Marl 80000: F1 0.13
- Sandstone 30000: F1 0.72
- Shale 65000: F1 0.65
- Sandstone/Shale 65030: F1 0.33
- Limestone 70000: F1 0.41

**But it struggles badly with rare classes:**
- Dolomite 74000: F1 ≈ 0
- Halite 88000: F1 = 0
- Basement 93000: F1 = 0
- Tuff 99000: F1 0.10



**confusion matrix**

In [6]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(y_train.unique())

cm = confusion_matrix(
    y_public,
    y_public_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print(cm_df)

         30000.0  65000.0  65030.0  70000.0  70032.0  74000.0  80000.0  \
30000.0    16967      543     1985     1076       63       58      704   
65000.0     2045    43731    13460     1150        6     1604    12363   
65030.0     2825     4550     6651      287       12      108      588   
70000.0      449      403      505     2372      106       31      631   
70032.0        0        0        0      511      112        0        2   
74000.0        6      164       51      102        0        3       61   
80000.0      406      117      290     1185       21        4     1200   
86000.0        1       38       30       18        0        0       11   
88000.0        0        0        0        0        0        0        0   
90000.0       55       27       94        4        0        0        1   
93000.0        0        0        0        0        0        0        0   
99000.0       93      205       18       18        0       32       56   

         86000.0  88000.0  90000.0  9

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

ENGINEERED_FILE = "../data/processed/engineered_features.parquet"

df = pd.read_parquet(ENGINEERED_FILE)

print("Shape:", df.shape)
print("Wells:", df["WELL"].nunique())

Shape: (1430974, 46)
Wells: 118


In [10]:
BASELINE_FEATURES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB"
]

ROLLING_FEATURES = [
    c for c in df.columns
    if "_ROLLMEAN_" in c or "_ROLLSTD_" in c
]

MISSING_FEATURES = [
    f"{feature}_MISSING"
    for feature in BASELINE_FEATURES
]

ENGINEERED_FEATURES = (
    BASELINE_FEATURES
    + ROLLING_FEATURES
    + ["RESISTIVITY_SEPARATION"]
    + MISSING_FEATURES
)

print("Baseline features:", len(BASELINE_FEATURES))
print("Rolling features:", len(ROLLING_FEATURES))
print("Resistivity feature:", 1)
print("Missingness features:", len(MISSING_FEATURES))
print("Total engineered features:", len(ENGINEERED_FEATURES))

missing = [f for f in ENGINEERED_FEATURES if f not in df.columns]
print("\nMissing columns:", missing)

Baseline features: 5
Rolling features: 30
Resistivity feature: 1
Missingness features: 5
Total engineered features: 41

Missing columns: []


just trying to reconstruc the official split just to check whether my feature selection works 

In [12]:
# Load the official well split
split_df = pd.read_csv("../data/splits/force2020_split.csv")

# Attach split to the engineered dataset
df = df.drop(columns=["SPLIT"], errors="ignore")

df = df.merge(
    split_df,
    on="WELL",
    how="left",
    validate="many_to_one"
)

print(df["SPLIT"].value_counts())
print()
print(df.groupby("SPLIT")["WELL"].nunique())

SPLIT
TRAIN         1171557
PUBLIC         136841
BLIND_TEST     122576
Name: count, dtype: int64

SPLIT
BLIND_TEST    10
PUBLIC        10
TRAIN         98
Name: WELL, dtype: int64


In [13]:
# Split the engineered dataset
train_eng = df[df["SPLIT"] == "TRAIN"].copy()
public_eng = df[df["SPLIT"] == "PUBLIC"].copy()
blind_eng = df[df["SPLIT"] == "BLIND_TEST"].copy()

# Training-only medians for all 41 engineered features
engineered_medians = train_eng[ENGINEERED_FEATURES].median()

print("Engineered features:", len(ENGINEERED_FEATURES))
print("Training rows:", len(train_eng))
print("Public rows:", len(public_eng))
print("Blind rows:", len(blind_eng))

print("\nFeatures with missing values before imputation:")
print(
    train_eng[ENGINEERED_FEATURES]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

Engineered features: 41
Training rows: 1171557
Public rows: 136841
Blind rows: 122576

Features with missing values before imputation:
RHOB                      161482
RHOB_ROLLMEAN_5           161479
RHOB_ROLLSTD_5            161479
RHOB_ROLLMEAN_9           161109
RHOB_ROLLSTD_9            161109
RHOB_ROLLSTD_21           160152
RHOB_ROLLMEAN_21          160152
DTC_ROLLMEAN_5             81470
DTC                        81470
DTC_ROLLSTD_5              81470
DTC_ROLLSTD_9              81238
DTC_ROLLMEAN_9             81238
DTC_ROLLSTD_21             80564
DTC_ROLLMEAN_21            80564
RESISTIVITY_SEPARATION     39693
dtype: int64


In [15]:
TARGET = "FORCE_2020_LITHOFACIES_LITHOLOGY"

print("Target:", TARGET)
print("Target exists:", TARGET in df.columns)

Target: FORCE_2020_LITHOFACIES_LITHOLOGY
Target exists: True


In [16]:
# Continuous engineered features only
CONTINUOUS_ENGINEERED_FEATURES = (
    BASELINE_FEATURES
    + ROLLING_FEATURES
    + ["RESISTIVITY_SEPARATION"]
)

# Apply TRAIN-derived medians to all splits
for feature in CONTINUOUS_ENGINEERED_FEATURES:
    median = engineered_medians[feature]

    train_eng[feature] = train_eng[feature].fillna(median)
    public_eng[feature] = public_eng[feature].fillna(median)
    blind_eng[feature] = blind_eng[feature].fillna(median)

# Construct X/y
X_train_eng = train_eng[ENGINEERED_FEATURES].copy()
y_train_eng = train_eng[TARGET].copy()

X_public_eng = public_eng[ENGINEERED_FEATURES].copy()
y_public_eng = public_eng[TARGET].copy()

X_blind_eng = blind_eng[ENGINEERED_FEATURES].copy()
y_blind_eng = blind_eng[TARGET].copy()

print("Remaining NaNs:")
print("TRAIN:", X_train_eng.isna().sum().sum())
print("PUBLIC:", X_public_eng.isna().sum().sum())
print("BLIND:", X_blind_eng.isna().sum().sum())

print("\nFeature shapes:")
print("TRAIN:", X_train_eng.shape)
print("PUBLIC:", X_public_eng.shape)
print("BLIND:", X_blind_eng.shape)

Remaining NaNs:
TRAIN: 0
PUBLIC: 0
BLIND: 0

Feature shapes:
TRAIN: (1171557, 41)
PUBLIC: (136841, 41)
BLIND: (122576, 41)


In [17]:
RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_feat

now trsining

In [18]:
rf_engineered = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

print("Training Random Forest with 41 features...")

start = time.perf_counter()

rf_engineered.fit(X_train_eng, y_train_eng)

engineered_train_time = time.perf_counter() - start

print(f"Training time: {engineered_train_time:.2f} seconds")

Training Random Forest with 41 features...
Training time: 172.27 seconds


**seconf evalution**

In [19]:
start = time.perf_counter()

y_public_eng_pred = rf_engineered.predict(X_public_eng)

engineered_prediction_time = time.perf_counter() - start

eng_accuracy = accuracy_score(y_public_eng, y_public_eng_pred)
eng_balanced_acc = balanced_accuracy_score(y_public_eng, y_public_eng_pred)
eng_macro_f1 = f1_score(y_public_eng, y_public_eng_pred, average="macro")
eng_weighted_f1 = f1_score(y_public_eng, y_public_eng_pred, average="weighted")

print(f"Prediction time:    {engineered_prediction_time:.2f} s")
print(f"Accuracy:            {eng_accuracy:.4f}")
print(f"Balanced Accuracy:  {eng_balanced_acc:.4f}")
print(f"Macro F1:            {eng_macro_f1:.4f}")
print(f"Weighted F1:         {eng_weighted_f1:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_public_eng,
        y_public_eng_pred,
        zero_division=0
    )
)

Prediction time:    0.98 s
Accuracy:            0.5740
Balanced Accuracy:  0.4458
Macro F1:            0.3042
Weighted F1:         0.6247

Classification report:
              precision    recall  f1-score   support

     30000.0       0.77      0.79      0.78     24048
     65000.0       0.89      0.56      0.69     84030
     65030.0       0.29      0.39      0.33     17558
     70000.0       0.38      0.60      0.47      4798
     70032.0       0.37      0.02      0.03       625
     74000.0       0.01      0.01      0.01       416
     80000.0       0.10      0.38      0.15      3306
     86000.0       0.75      0.12      0.21       125
     88000.0       0.00      0.00      0.00         0
     90000.0       0.41      0.87      0.56       690
     99000.0       0.07      0.73      0.13      1245

    accuracy                           0.57    136841
   macro avg       0.37      0.41      0.30    136841
weighted avg       0.74      0.57      0.62    136841



/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Metric | 5 features | 41 features | Change |
|---|---:|---:|---:|
| Accuracy | 0.5289 | **0.5740** | **+0.0451** |
| Balanced Accuracy | 0.4237 | **0.4458** | **+0.0221** |
| Macro F1 | 0.2718 | **0.3042** | **+0.0324** |
| Weighted F1 | 0.5922 | **0.6247** | **+0.0325** |

Several classes improved substantially:
- Sandstone: 0.72 → 0.78
- Shale: 0.65 → 0.69
- Limestone: 0.41 → 0.47
- Coal: 0.37 → 0.56

But some remain very difficult:
- Chalk: 0.24 → 0.03
- Dolomite: 0.00 → 0.01
- Marl: 0.13 → 0.15
- Tuff: 0.10 → 0.13

baseline + missingness indicators

5 baseline logs + 5 missing indicators = 10 features
We'll keep the exact same Random Forest configuration and evaluate only on PUBLIC

In [20]:
ABLATION_A_FEATURES = BASELINE_FEATURES + MISSING_FEATURES

X_train_A = train_eng[ABLATION_A_FEATURES].copy()
X_public_A = public_eng[ABLATION_A_FEATURES].copy()

print("Features:", len(ABLATION_A_FEATURES))
print(ABLATION_A_FEATURES)

rf_A = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

start = time.perf_counter()

rf_A.fit(X_train_A, y_train_eng)

y_public_A = rf_A.predict(X_public_A)

time_A = time.perf_counter() - start

A_accuracy = accuracy_score(y_public_eng, y_public_A)
A_balanced = balanced_accuracy_score(y_public_eng, y_public_A)
A_macro_f1 = f1_score(y_public_eng, y_public_A, average="macro")
A_weighted_f1 = f1_score(y_public_eng, y_public_A, average="weighted")

print("\nResults:")
print(f"Accuracy:           {A_accuracy:.4f}")
print(f"Balanced Accuracy:  {A_balanced:.4f}")
print(f"Macro F1:           {A_macro_f1:.4f}")
print(f"Weighted F1:        {A_weighted_f1:.4f}")
print(f"Train + prediction: {time_A:.2f} s")

Features: 10
['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB', 'GR_MISSING', 'RDEP_LOG10_MISSING', 'RMED_LOG10_MISSING', 'DTC_MISSING', 'RHOB_MISSING']

Results:
Accuracy:           0.5288
Balanced Accuracy:  0.4714
Macro F1:           0.3036
Weighted F1:        0.5948
Train + prediction: 67.99 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Metric | 5 baseline | + Missing indicators | Change |
|---|---:|---:|---:|
| Accuracy | 0.5289 | 0.5288 | −0.0001 |
| Balanced Accuracy | 0.4237 | **0.4714** | **+0.0477** |
| Macro F1 | 0.2718 | **0.3036** | **+0.0318** |
| Weighted F1 | 0.5922 | 0.5948 | +0.0026 |

Baseline + resistivity separation

In [21]:
ABLATION_B_FEATURES = BASELINE_FEATURES + ["RESISTIVITY_SEPARATION"]

X_train_B = train_eng[ABLATION_B_FEATURES].copy()
X_public_B = public_eng[ABLATION_B_FEATURES].copy()

rf_B = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

start = time.perf_counter()

rf_B.fit(X_train_B, y_train_eng)

y_public_B = rf_B.predict(X_public_B)

time_B = time.perf_counter() - start

B_accuracy = accuracy_score(y_public_eng, y_public_B)
B_balanced = balanced_accuracy_score(y_public_eng, y_public_B)
B_macro_f1 = f1_score(y_public_eng, y_public_B, average="macro")
B_weighted_f1 = f1_score(y_public_eng, y_public_B, average="weighted")

print("Features:", len(ABLATION_B_FEATURES))
print(f"Accuracy:           {B_accuracy:.4f}")
print(f"Balanced Accuracy:  {B_balanced:.4f}")
print(f"Macro F1:           {B_macro_f1:.4f}")
print(f"Weighted F1:        {B_weighted_f1:.4f}")
print(f"Train + prediction: {time_B:.2f} s")

Features: 6
Accuracy:           0.5410
Balanced Accuracy:  0.4389
Macro F1:           0.2827
Weighted F1:        0.6033
Train + prediction: 80.37 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Feature set | Features | Accuracy | Balanced Acc. | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|---:|
| Baseline | 5 | 0.5289 | 0.4237 | 0.2718 | 0.5922 |
| + Missing indicators | 10 | 0.5288 | **0.4714** | **0.3036** | 0.5948 |
| + Resistivity separation | 6 | 0.5410 | 0.4389 | 0.2827 | **0.6033** |
| All engineered | 41 | **0.5740** | 0.4458 | **0.3042** | **0.6247** |

The two feature groups behave differently:
Missingness indicators
- Huge improvement in balanced accuracy: 0.4237 → 0.4714
- Macro-F1: 0.2718 → 0.3036
- Almost no accuracy improvement.

So missingness carries useful class-discriminative information.
Resistivity separation
- Accuracy: 0.5289 → 0.5410
- Macro-F1: 0.2718 → 0.2827

So it provides a smaller but consistent improvement.

**Let's isolate the rolling features:**
 Baseline + rolling statistics = 35 features

In [23]:
ABLATION_C_FEATURES = BASELINE_FEATURES + ROLLING_FEATURES

X_train_C = train_eng[ABLATION_C_FEATURES].copy()
X_public_C = public_eng[ABLATION_C_FEATURES].copy()

print("Number of features:", len(ABLATION_C_FEATURES))

rf_C = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

print("Training...")

start = time.perf_counter()

rf_C.fit(X_train_C, y_train_eng)

y_public_C = rf_C.predict(X_public_C)

time_C = time.perf_counter() - start

C_accuracy = accuracy_score(y_public_eng, y_public_C)
C_balanced = balanced_accuracy_score(y_public_eng, y_public_C)
C_macro_f1 = f1_score(y_public_eng, y_public_C, average="macro")
C_weighted_f1 = f1_score(y_public_eng, y_public_C, average="weighted")

print("\nResults:")
print(f"Accuracy:           {C_accuracy:.4f}")
print(f"Balanced Accuracy:  {C_balanced:.4f}")
print(f"Macro F1:           {C_macro_f1:.4f}")
print(f"Weighted F1:        {C_weighted_f1:.4f}")
print(f"Train + prediction: {time_C:.2f} s")

Number of features: 35
Training...

Results:
Accuracy:           0.5735
Balanced Accuracy:  0.4539
Macro F1:           0.3157
Weighted F1:        0.6229
Train + prediction: 158.04 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Feature set | Features | Accuracy | Balanced Acc. | Macro F1 | Weighted F1 |
|---|---:|---:|---:|---:|---:|
| Baseline | 5 | 0.5289 | 0.4237 | 0.2718 | 0.5922 |
| + Missing indicators | 10 | 0.5288 | **0.4714** | 0.3036 | 0.5948 |
| + Resistivity separation | 6 | 0.5410 | 0.4389 | 0.2827 | 0.6033 |
| **+ Rolling features** | **35** | **0.5735** | **0.4539** | **0.3157** | **0.6229** |
| All engineered | 41 | **0.5740** | 0.4458 | 0.3042 | **0.6247** |

**extra tree training on all the 46 features**

In [24]:
from sklearn.ensemble import ExtraTreesClassifier
import time

X_train = train_eng[ENGINEERED_FEATURES].copy()
X_public = public_eng[ENGINEERED_FEATURES].copy()

print("Number of features:", X_train.shape[1])

et = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

print("Training...")

start = time.perf_counter()

et.fit(X_train, y_train_eng)

y_public_et = et.predict(X_public)

time_et = time.perf_counter() - start

accuracy_et = accuracy_score(y_public_eng, y_public_et)
balanced_acc_et = balanced_accuracy_score(y_public_eng, y_public_et)
macro_f1_et = f1_score(y_public_eng, y_public_et, average="macro", zero_division=0)
weighted_f1_et = f1_score(y_public_eng, y_public_et, average="weighted", zero_division=0)

print("\nExtra Trees Results:")
print(f"Accuracy:           {accuracy_et:.4f}")
print(f"Balanced Accuracy:  {balanced_acc_et:.4f}")
print(f"Macro F1:           {macro_f1_et:.4f}")
print(f"Weighted F1:        {weighted_f1_et:.4f}")
print(f"Train + prediction: {time_et:.2f} s")

Number of features: 41
Training...

Extra Trees Results:
Accuracy:           0.5439
Balanced Accuracy:  0.5092
Macro F1:           0.3432
Weighted F1:        0.6081
Train + prediction: 37.71 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Model / Feature set | Features | Accuracy | Balanced Acc. | Macro F1 | Weighted F1 | Time |
|---|---:|---:|---:|---:|---:|---:|
| RF — Baseline | 5 | 0.5289 | 0.4237 | 0.2718 | 0.5922 | — |
| RF — + Missing | 10 | 0.5288 | 0.4714 | 0.3036 | 0.5948 | 68s |
| RF — + Resistivity | 6 | 0.5410 | 0.4389 | 0.2827 | 0.6033 | 80s |
| RF — + Rolling | 35 | 0.5735 | 0.4539 | 0.3157 | 0.6229 | 158s |
| **RF — All engineered** | **41** | **0.5740** | **0.4458** | **0.3042** | **0.6247** | **172s** |
| **Extra Trees — All engineered** | **41** | **0.5439** | **0.5092** | **0.3432** | **0.6081** | **37.7s** |

**Extra Trees has lower overall accuracy than RF:**
54.39% vs 57.40%

But it has much better class-balanced performance:
Balanced Accuracy: 50.92% vs 44.58%

and
Macro F1: 34.32% vs 30.42%

That's actually quite significant for your lithology problem because the dataset is heavily imbalanced. \n
Extra Trees is also dramatically faster:
37.7 s vs 172.3 s

**XGBoost**

In [27]:
from xgboost import XGBClassifier
import time

X_train = train_eng[ENGINEERED_FEATURES].copy()
X_public = public_eng[ENGINEERED_FEATURES].copy()

# Convert labels to 0...n_classes-1 for XGBoost
classes = sorted(y_train_eng.unique())
class_to_int = {c: i for i, c in enumerate(classes)}
int_to_class = {i: c for c, i in class_to_int.items()}

y_train_xgb = y_train_eng.map(class_to_int)
y_public_xgb = y_public_eng.map(class_to_int)

print("Number of features:", X_train.shape[1])
print("Number of classes:", len(classes))

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(classes),
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print("Training...")

start = time.perf_counter()

xgb_model.fit(X_train, y_train_xgb)

y_pred_xgb_int = xgb_model.predict(X_public)
y_public_pred_xgb = pd.Series(
    y_pred_xgb_int,
    index=y_public_eng.index
).map(int_to_class)

time_xgb = time.perf_counter() - start

accuracy_xgb = accuracy_score(y_public_eng, y_public_pred_xgb)
balanced_acc_xgb = balanced_accuracy_score(y_public_eng, y_public_pred_xgb)
macro_f1_xgb = f1_score(
    y_public_eng, y_public_pred_xgb,
    average="macro",
    zero_division=0
)
weighted_f1_xgb = f1_score(
    y_public_eng, y_public_pred_xgb,
    average="weighted",
    zero_division=0
)

print("\nXGBoost Results:")
print(f"Accuracy:           {accuracy_xgb:.4f}")
print(f"Balanced Accuracy:  {balanced_acc_xgb:.4f}")
print(f"Macro F1:           {macro_f1_xgb:.4f}")
print(f"Weighted F1:        {weighted_f1_xgb:.4f}")
print(f"Train + prediction: {time_xgb:.2f} s")

Number of features: 41
Number of classes: 12
Training...

XGBoost Results:
Accuracy:           0.7545
Balanced Accuracy:  0.3318
Macro F1:           0.3151
Weighted F1:        0.7217
Train + prediction: 90.21 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Model | Accuracy | Balanced Accuracy | Macro F1 | Weighted F1 | Time |
|---|---:|---:|---:|---:|---:|
| Random Forest | **0.5740** | 0.4458 | 0.3042 | 0.6247 | 172 s |
| Extra Trees | 0.5439 | **0.5092** | **0.3432** | 0.6081 | **37.7 s** |
| **XGBoost** | **0.7545** | 0.3318 | 0.3151 | **0.7217** | 90.2 s |

XGBoost has the highest accuracy by far, but its:

Balanced Accuracy = 33.18%

is actually the worst of the three.

This strongly suggests XGBoost is doing very well on the dominant lithologies—especially Shale (65000)—while performing poorly on several minority lithologies.

That's exactly why we are tracking multiple metrics.
For your project, I'd currently rank them:


Macro F1 / Balanced Accuracy → primary model-selection metrics

Weighted F1 / Accuracy → secondary metrics
because your lithology distribution is extremely imbalanced.

In [28]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_public_eng,
        y_public_pred_xgb,
        zero_division=0
    )
)

              precision    recall  f1-score   support

     30000.0       0.81      0.81      0.81     24048
     65000.0       0.81      0.93      0.86     84030
     65030.0       0.32      0.14      0.19     17558
     70000.0       0.41      0.48      0.45      4798
     70032.0       0.02      0.00      0.00       625
     74000.0       0.00      0.00      0.00       416
     80000.0       0.19      0.09      0.12      3306
     86000.0       0.15      0.02      0.03       125
     88000.0       0.00      0.00      0.00         0
     90000.0       0.68      0.59      0.63       690
     99000.0       0.63      0.27      0.38      1245

    accuracy                           0.75    136841
   macro avg       0.37      0.30      0.32    136841
weighted avg       0.71      0.75      0.72    136841



| Lithology | Recall | F1 | Interpretation |
|---|---:|---:|---|
| Sandstone | 0.81 | 0.81 | Good |
| **Shale** | **0.93** | **0.86** | Excellent, but dominant class |
| Sandstone/Shale | 0.14 | 0.19 | Poor |
| Limestone | 0.48 | 0.45 | Moderate |
| Chalk | 0.00 | 0.00 | Failed |
| Dolomite | 0.00 | 0.00 | Failed |
| Marl | 0.09 | 0.12 | Poor |
| Anhydrite | 0.02 | 0.03 | Very poor |
| Coal | 0.59 | 0.63 | Reasonable |
| Tuff | 0.27 | 0.38 | Poor |

****CatBoost****

In [29]:
from catboost import CatBoostClassifier
import time

X_train = train_eng[ENGINEERED_FEATURES].copy()
X_public = public_eng[ENGINEERED_FEATURES].copy()

print("Number of features:", X_train.shape[1])

cat_model = CatBoostClassifier(
    iterations=100,
    depth=8,
    learning_rate=0.1,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    verbose=False,
    thread_count=-1,
    random_seed=42
)

print("Training...")

start = time.perf_counter()

cat_model.fit(X_train, y_train_eng)

y_public_cat = cat_model.predict(X_public).ravel()

time_cat = time.perf_counter() - start

accuracy_cat = accuracy_score(y_public_eng, y_public_cat)
balanced_acc_cat = balanced_accuracy_score(y_public_eng, y_public_cat)
macro_f1_cat = f1_score(
    y_public_eng, y_public_cat,
    average="macro",
    zero_division=0
)
weighted_f1_cat = f1_score(
    y_public_eng, y_public_cat,
    average="weighted",
    zero_division=0
)

print("\nCatBoost Results:")
print(f"Accuracy:           {accuracy_cat:.4f}")
print(f"Balanced Accuracy:  {balanced_acc_cat:.4f}")
print(f"Macro F1:           {macro_f1_cat:.4f}")
print(f"Weighted F1:        {weighted_f1_cat:.4f}")
print(f"Train + prediction: {time_cat:.2f} s")

Number of features: 41
Training...

CatBoost Results:
Accuracy:           0.7556
Balanced Accuracy:  0.3125
Macro F1:           0.3054
Weighted F1:        0.7221
Train + prediction: 437.59 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Model | Accuracy ↑ | Balanced Accuracy ↑ | Macro F1 ↑ | Weighted F1 ↑ | Time ↓ |
|---|---:|---:|---:|---:|---:|
| Random Forest | 0.5740 | 0.4458 | 0.3042 | 0.6247 | 172 s |
| **Extra Trees** | 0.5439 | **0.5092** | **0.3432** | 0.6081 | **37.7 s** |
| XGBoost | 0.7545 | 0.3318 | 0.3151 | **0.7217** | 90.2 s |
| CatBoost | **0.7556** | 0.3125 | 0.3054 | **0.7221** | 437.6 s |

There are three very different behaviors:
1. XGBoost/CatBoost → excellent overall accuracy
~75.5% accuracy and ~72.2% weighted F1.
But they heavily favor the dominant lithologies, giving poor:
- Balanced Accuracy
- Macro F1
2. Extra Trees → best balanced classifier
Extra Trees has:
Balanced Accuracy = 50.92%
Macro F1 = 34.32%

Both are currently the best.
3. CatBoost → not attractive in this configuration
It took:
437.6 seconds ≈ 7.3 minutes

while giving the worst balanced accuracy (31.25%).

**class balanced xgboost**

In [30]:
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

# Calculate weights ONLY from training labels
sample_weights_xgb = compute_sample_weight(
    class_weight="balanced",
    y=y_train_eng
)

print("Sample weights:")
print("  Minimum:", sample_weights_xgb.min())
print("  Maximum:", sample_weights_xgb.max())
print("  Mean:   ", sample_weights_xgb.mean())

# Check average weight per lithology
weight_summary = (
    pd.DataFrame({
        "lithology": y_train_eng.values,
        "weight": sample_weights_xgb
    })
    .groupby("lithology")["weight"]
    .mean()
)

print("\nAverage weight per lithology:")
print(weight_summary)

Sample weights:
  Minimum: 0.1353740718400965
  Maximum: 692.4095744680851
  Mean:    1.0

Average weight per lithology:
lithology
30000.0      0.576310
65000.0      0.135374
65030.0      0.648250
70000.0      1.733483
70032.0      9.286574
74000.0     57.837530
80000.0      2.928658
86000.0     89.981336
88000.0     11.885774
90000.0     25.557526
93000.0    692.409574
99000.0      6.404051
Name: weight, dtype: float64


In [31]:
xgb_balanced = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(classes),
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print("Training balanced XGBoost...")

start = time.perf_counter()

xgb_balanced.fit(
    X_train,
    y_train_xgb,
    sample_weight=sample_weights_xgb
)

y_pred_balanced_int = xgb_balanced.predict(X_public)

y_public_pred_xgb_balanced = pd.Series(
    y_pred_balanced_int,
    index=y_public_eng.index
).map(int_to_class)

time_xgb_balanced = time.perf_counter() - start

print("\nBalanced XGBoost Results:")
print(f"Accuracy:           {accuracy_score(y_public_eng, y_public_pred_xgb_balanced):.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_score(y_public_eng, y_public_pred_xgb_balanced):.4f}")
print(f"Macro F1:           {f1_score(y_public_eng, y_public_pred_xgb_balanced, average='macro', zero_division=0):.4f}")
print(f"Weighted F1:        {f1_score(y_public_eng, y_public_pred_xgb_balanced, average='weighted', zero_division=0):.4f}")
print(f"Train + prediction: {time_xgb_balanced:.2f} s")

Training balanced XGBoost...

Balanced XGBoost Results:
Accuracy:           0.5416
Balanced Accuracy:  0.4662
Macro F1:           0.2934
Weighted F1:        0.6047
Train + prediction: 71.42 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| Model | Accuracy | Balanced Accuracy | Macro F1 | Weighted F1 | Time |
|---|---:|---:|---:|---:|---:|
| RF — 41 features | 0.5740 | 0.4458 | 0.3042 | 0.6247 | 172 s |
| **Extra Trees — 41** | 0.5439 | **0.5092** | **0.3432** | 0.6081 | **37.7 s** |
| XGBoost — unweighted | **0.7545** | 0.3318 | 0.3151 | **0.7217** | 90.2 s |
| XGBoost — balanced | 0.5416 | 0.4662 | 0.2934 | 0.6047 | 71.4 s |
| CatBoost — unweighted | 0.7556 | 0.3125 | 0.3054 | 0.7221 | 437.6 s |

In [32]:
print("EXTRA TREES — classification report\n")

print(
    classification_report(
        y_public_eng,
        y_public_et,
        zero_division=0
    )
)

EXTRA TREES — classification report

              precision    recall  f1-score   support

     30000.0       0.77      0.77      0.77     24048
     65000.0       0.89      0.51      0.65     84030
     65030.0       0.35      0.37      0.36     17558
     70000.0       0.43      0.55      0.49      4798
     70032.0       0.38      0.20      0.26       625
     74000.0       0.04      0.14      0.06       416
     80000.0       0.09      0.54      0.16      3306
     86000.0       0.75      0.30      0.43       125
     88000.0       0.00      0.00      0.00         0
     90000.0       0.33      0.93      0.48       690
     99000.0       0.06      0.76      0.11      1245

    accuracy                           0.54    136841
   macro avg       0.37      0.46      0.34    136841
weighted avg       0.75      0.54      0.61    136841



| Lithology | ET F1 | XGB F1 | ET recall | XGB recall |
|---|---:|---:|---:|---:|
| Sandstone | **0.77** | 0.81 | 0.77 | **0.81** |
| Shale | 0.65 | **0.86** | 0.51 | **0.93** |
| Sandstone/Shale | **0.36** | 0.19 | **0.37** | 0.14 |
| Limestone | **0.49** | 0.45 | **0.55** | 0.48 |
| Chalk | **0.26** | 0.00 | **0.20** | 0.00 |
| Dolomite | **0.06** | 0.00 | **0.14** | 0.00 |
| Marl | **0.16** | 0.12 | **0.54** | 0.09 |
| Anhydrite | **0.43** | 0.03 | **0.30** | 0.02 |
| Coal | 0.48 | **0.63** | **0.93** | 0.59 |
| Tuff | 0.11 | **0.38** | **0.76** | 0.27 |

In [33]:
xgb_300 = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(classes),
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print("Training XGBoost (300 trees, depth 3)...")

start = time.perf_counter()

xgb_300.fit(X_train, y_train_xgb)

y_pred_300_int = xgb_300.predict(X_public)

y_public_pred_xgb_300 = pd.Series(
    y_pred_300_int,
    index=y_public_eng.index
).map(int_to_class)

time_xgb_300 = time.perf_counter() - start

print("\nXGBoost 300/3 Results:")
print(f"Accuracy:           {accuracy_score(y_public_eng, y_public_pred_xgb_300):.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_score(y_public_eng, y_public_pred_xgb_300):.4f}")
print(f"Macro F1:           {f1_score(y_public_eng, y_public_pred_xgb_300, average='macro', zero_division=0):.4f}")
print(f"Weighted F1:        {f1_score(y_public_eng, y_public_pred_xgb_300, average='weighted', zero_division=0):.4f}")
print(f"Train + prediction: {time_xgb_300:.2f} s")

Training XGBoost (300 trees, depth 3)...

XGBoost 300/3 Results:
Accuracy:           0.7497
Balanced Accuracy:  0.3302
Macro F1:           0.3185
Weighted F1:        0.7164
Train + prediction: 199.50 s


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


| XGBoost | Accuracy | Balanced Acc. | Macro F1 | Weighted F1 | Time |
|---|---:|---:|---:|---:|---:|
| 100 trees, depth 6 | **0.7545** | **0.3318** | 0.3151 | **0.7217** | 90 s |
| 300 trees, depth 3 | 0.7497 | 0.3302 | **0.3185** | 0.7164 | 199.5 s |

So:
- Accuracy: essentially unchanged
- Balanced accuracy: essentially unchanged
- Macro F1: tiny improvement, 0.3151 → 0.3185
- Training time: more than 2×
- Still nowhere near Extra Trees' 0.3432 Macro F1 / 0.5092 balanced accuracy

This is an important conclusion
Our problem is not simply insufficient number/depth of XGBoost trees.
And I would stop spending compute on arbitrary XGBoost configurations at this point.


In [ ]:
xgb_w1 = XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1500,
    objective="multi:softprob",
    num_class=len(classes),
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print("Training W1 XGBoost...")

start = time.perf_counter()

xgb_w1.fit(X_train, y_train_xgb)

# softprob returns class probabilities; convert to class index
y_prob_w1 = xgb_w1.predict_proba(X_public)
y_pred_w1_int = np.argmax(y_prob_w1, axis=1)

y_public_pred_w1 = pd.Series(
    y_pred_w1_int,
    index=y_public_eng.index
).map(int_to_class)

time_w1 = time.perf_counter() - start

print("\nW1 XGBoost Results:")
print(f"Accuracy:           {accuracy_score(y_public_eng, y_public_pred_w1):.4f}")
print(f"Balanced Accuracy:  {balanced_accuracy_score(y_public_eng, y_public_pred_w1):.4f}")
print(f"Macro F1:           {f1_score(y_public_eng, y_public_pred_w1, average='macro', zero_division=0):.4f}")
print(f"Weighted F1:        {f1_score(y_public_eng, y_public_pred_w1, average='weighted', zero_division=0):.4f}")
print(f"Train + prediction: {time_w1:.2f} s")

Training W1 XGBoost...
